In [1]:
from pymatgen.core.structure import Structure

# import dask

import time
import shutil
import yaml
import subprocess
import os.path, os
from pathlib import Path
import pandas as pd
import ase
import ase.io
import ase.io.espresso
from ase.data import atomic_masses, atomic_numbers
import json

from util_qe import get_k_point_density, get_q_point_density, \
        update_dict_relax, \
        update_dict_ph_elph, \
        update_q2r_matdyn_elph, \
        write_qe_file, \
        get_k_q_grid


In [2]:
par_el = 'run_full'
root = '/blue/hennig/jasongibson/diff_model'
root = f'{root}/materials/{par_el}/mp_relaxed/'
df = pd.read_pickle(f'pkl_files/df_{par_el}_pred_m3gnet_eah.pkl')

In [3]:
df.shape

(6173, 28)

In [4]:
def submit_qe_cal(jobdir):
    print(jobdir)
    k_point_density = get_k_point_density() #min kpoints per inv A
    q_point_density = get_q_point_density() #min qpoints per inv A


    if not Path(jobdir+"/relax.out").is_file():
        filename = jobdir+"/CONTCAR"

        atom_obj = ase.io.read(filename)

        cell = atom_obj.cell
        reciprocal_cell = cell.reciprocal()
        k_grid, q_grid = get_k_q_grid(k_point_density,q_point_density,reciprocal_cell)


        with open("default_qe_para.json","r") as fp:
            para = json.load(fp)
        relax_parameter = para["relax_parameter"]

        relax_parameter = update_dict_relax(relax_parameter,k_grid,atom_obj)
        write_qe_file(param=relax_parameter,filename=jobdir+"/relax.in")

    filename = jobdir+"/relax.out"
    atom_obj = ase.io.read(filename)
    # os.system("rm -r "+jobdir+"/temp")

    if not Path(jobdir+"/phonon_2x2x2/").is_dir():
        os.mkdir(jobdir+"/phonon_2x2x2/")
    jobdir = jobdir+"/phonon_2x2x2/."

    cell = atom_obj.cell
    reciprocal_cell = cell.reciprocal()
    k_grid, q_grid = get_k_q_grid(k_point_density,q_point_density,reciprocal_cell)

    #devnull = open(os.path.join(jobdir,'job.log'), 'w')

    with open("default_qe_para.json","r") as fp:
        para = json.load(fp)

    relax_parameter = para["relax_parameter"]
    relax_parameter = update_dict_relax(relax_parameter,k_grid,atom_obj)
    
    scf_dense = relax_parameter.copy()
    scf_dense["CONTROL"]["calculation"] = "scf"
    scf_dense["ELECTRONS"]["conv_thr"] = 1.0e-12
    scf_coarse = scf_dense.copy()

    write_qe_file(param=scf_dense,filename=jobdir+"/scf_dense.in")

    ph_elph = para["ph"]
    q2r = para["q2r"]
    matdyn = para["matdyn_dos"]

    ph_elph = update_dict_ph_elph(ph_elph, q_grid,atom_obj)
    ph_elph["INPUTPH"]["nmix_ph"] = 20
    ph_elph["INPUTPH"]["diagonalization"] = "cg"
    ph_elph["INPUTPH"]["nq1"] = 2
    ph_elph["INPUTPH"]["nq2"] = 2
    ph_elph["INPUTPH"]["nq3"] = 2
    ph_elph["INPUTPH"].pop("electron_phonon")
    ph_elph["INPUTPH"].pop("el_ph_sigma")
    ph_elph["INPUTPH"].pop("el_ph_nsigma")
    ph_elph["INPUTPH"]['recover'] = True

    q2r, matdyn = update_q2r_matdyn_elph(q2r, matdyn, q_grid,atom_obj)
    matdyn["input"].pop("la2F")
    matdyn["input"].pop("el_ph_nsigma")
    q2r["input"].pop("la2F")
    q2r["input"].pop("el_ph_nsigma")

    write_qe_file(param=ph_elph,filename=jobdir+"/phonon_2x2x2.in")
    write_qe_file(param=q2r,filename=jobdir+"/q2r.in")
    write_qe_file(param=matdyn,filename=jobdir+"/matdyn.in")



In [5]:
def check_second_to_last_line(file_path):
    with open(file_path, 'r') as file:
        lines = file.readlines()
        second_to_last_line = lines[-2].strip()
        
    return second_to_last_line == 'JOB DONE.'

In [6]:
list_uniq = df.index.values

In [7]:
done = []
not_done = []
for line in list_uniq:
    if os.path.isfile(root+f'{line}'+f'/relax.out'):
        try:
            done.append(check_second_to_last_line(root+f'{line}/relax.out'))
            if done[-1] == False:
                not_done.append(line)
        except:
            done.append(False)
            not_done.append(line)
    else:
        done.append(False)
        not_done.append(line)
            

In [12]:
# done = []
# not_done = []
# for line in list_uniq:
#     try:
#         done.append(check_second_to_last_line(root+line+f'/relax.out'))
#     except:
#         done.append(check_second_to_last_line(root+line.split('/')[0]+f'/relax.out'))
        

In [13]:
list_done = []
for t,i in zip(done,list_uniq):
    if t:
        list_done.append(i)


In [26]:
with open(root+'list_ele', "w") as file:
    for item in list_done:
        # Write each item to the file followed by a newline
        # file.write(f"{item}\n")        
        file.write(f"{item}/phonon_2x2x2\n")

In [18]:
from tqdm.notebook import tqdm

In [25]:
relax = []
for line in tqdm(list_done):
    try:
        submit_qe_cal(root+str(line))
        relax.append(line)        
    except:
        print(line)

  0%|          | 0/4351 [00:00<?, ?it/s]

/blue/hennig/jasongibson/diff_model/materials/run_full/mp_relaxed/55598
/blue/hennig/jasongibson/diff_model/materials/run_full/mp_relaxed/67960
/blue/hennig/jasongibson/diff_model/materials/run_full/mp_relaxed/124523
/blue/hennig/jasongibson/diff_model/materials/run_full/mp_relaxed/175289
175289
/blue/hennig/jasongibson/diff_model/materials/run_full/mp_relaxed/128096
/blue/hennig/jasongibson/diff_model/materials/run_full/mp_relaxed/5759
/blue/hennig/jasongibson/diff_model/materials/run_full/mp_relaxed/10289
/blue/hennig/jasongibson/diff_model/materials/run_full/mp_relaxed/182341
/blue/hennig/jasongibson/diff_model/materials/run_full/mp_relaxed/38814
/blue/hennig/jasongibson/diff_model/materials/run_full/mp_relaxed/64489
/blue/hennig/jasongibson/diff_model/materials/run_full/mp_relaxed/75570
/blue/hennig/jasongibson/diff_model/materials/run_full/mp_relaxed/3262
/blue/hennig/jasongibson/diff_model/materials/run_full/mp_relaxed/15149
/blue/hennig/jasongibson/diff_model/materials/run_full/

In [77]:
def submit_relax(jobdir):
    k_point_density = get_k_point_density() #min kpoints per inv A
    q_point_density = get_q_point_density() #min qpoints per inv A


    filename = jobdir+"/CONTCAR"

    atom_obj = ase.io.read(filename)

    cell = atom_obj.cell
    reciprocal_cell = cell.reciprocal()
    k_grid, q_grid = get_k_q_grid(k_point_density,q_point_density,reciprocal_cell)


    with open("default_qe_para.json","r") as fp:
        para = json.load(fp)
    relax_parameter = para["relax_parameter"]

    relax_parameter = update_dict_relax(relax_parameter,k_grid,atom_obj)
    write_qe_file(param=relax_parameter,filename=jobdir+"/relax.in")

In [85]:
relax_good = []
for line in tqdm(relax):
    try:
        submit_relax(root+str(line))
        relax_good.append(line)
        
    except:
        print(line)

  0%|          | 0/36 [00:00<?, ?it/s]

94843
285504


In [87]:
with open(root+'list_ele', "w") as file:
    for item in relax_good:
        # Write each item to the file followed by a newline
        file.write(f"{item}\n")
        # file.write(f"{item}/phonon_2x2x2\n")